In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer


import warnings
warnings.filterwarnings('ignore')


In [ ]:
tracks = pd.read_csv('tracks_records.csv')
tracks.head()


In [ ]:
tracks.shape


This is to check the information of data.Like data type of the colom

In [ ]:
tracks.info()

This is to check the sum of nullvalues of dataset foreach colom

In [ ]:
tracks.isnull().sum()

This is for removing nan rows because we have 170653 rows if we remove sum rows also no problem.But  we dont have any nan values.If there is any nan values we can watch in bar graph.

In [ ]:
tracks.dropna(inplace = True)
tracks.isnull().sum().plot.bar()
plt.show()

In [ ]:
tracks = tracks.drop(['id'], axis =1)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder

visual_tracks = pd.read_csv('tracks_records.csv')

# Drop rows with missing values
visual_tracks.dropna(inplace=True)

# Drop the 'id' and 'name' columns
visual_tracks.drop(columns=['id', 'name'], inplace=True)

# Initialize label encoder
label_encoder = LabelEncoder()

# Extracting and joining artists' names
visual_tracks['artists'] = visual_tracks['artists'].apply(lambda x: x.strip("[]").replace("'", ""))
artists_encoded = label_encoder.fit_transform(visual_tracks['artists'])
# Drop the original 'artists' column
visual_tracks.drop(columns=['artists'], inplace=True)

# Assign encoded artists to the dataset
visual_tracks['artists_encoded'] = artists_encoded

# Drop 'release_date' column
visual_tracks.drop(columns=['release_date'], inplace=True)

# Initialize TSNE model
model = TSNE(n_components=2, random_state=0)

# Fit and transform the data
tsne_data = model.fit_transform(visual_tracks.head(500))

# Plot the t-SNE visualization
plt.figure(figsize=(7, 7))
plt.scatter(tsne_data[:, 0], tsne_data[:, 1])
plt.show()

In [ ]:
tracks['name'].nunique(), tracks.shape

In [ ]:
tracks = tracks.sort_values(by=['popularity'], ascending=False)
tracks.drop_duplicates(subset=['name'], keep='first', inplace=True)
tracks.shape

In [ ]:
# plt.figure(figsize = (8, 5))
# tracks['Start Date']=pd.to_datetime(tracks['Start Date'])
# sb.countplot(tracks['release_date'])
# plt.axis('off')
# plt.show()

In [ ]:
floats = []
for col in tracks.columns:
  if tracks[col].dtype == 'float':
    floats.append(col)
 
len(floats)

In [ ]:
plt.subplots(figsize = (15, 5))
for i, col in enumerate(floats):
  plt.subplot(2, 5, i + 1)
  sb.distplot(tracks[col])
plt.tight_layout()
plt.show()

In [ ]:
%%capture
song_vectorizer = CountVectorizer()
song_vectorizer.fit(tracks['name'])

In [ ]:
tracks = tracks.sort_values(by=['popularity'], ascending=False).head(10000)
tracks.shape

In [ ]:

def get_similarities(song_name, data):
   
  # Getting vector for the input song.
  text_array1 = song_vectorizer.transform(data[data['name']==song_name]['name']).toarray()
  num_array1 = data[data['name']==song_name].select_dtypes(include=np.number).to_numpy()
   
  # We will store similarity for each row of the dataset
  sim = []
  for idx, row in data.iterrows():
    name = row['name']
     
    # Getting vector for current song.
    text_array2 = song_vectorizer.transform(data[data['name']==name]['name']).toarray()
    num_array2 = data[data['name']==name].select_dtypes(include=np.number).to_numpy()
 
    # Calculating similarities for text as well as numeric features
    text_sim = cosine_similarity(text_array1, text_array2)[0][0]
    num_sim = cosine_similarity(num_array1, num_array2)[0][0]
    sim.append(text_sim + num_sim)
     
  return sim





In [ ]:
def recommend_songs(song_name, data=tracks):
  # Base case
  if tracks[tracks['name'] == song_name].shape[0] == 0:
    print('This song is either not so popular or you\
    have entered invalid_name.\n Some songs you may like:\n')
     
    for song in data.sample(n=5)['name'].values:
      print(song)
    return
   
  data['similarity_factor'] = get_similarities(song_name, data)
 
  data.sort_values(by=['similarity_factor', 'popularity','year'],
                   ascending = [False, False,False],
                   inplace=True)
   
  # First song will be the input song itself as the similarity will be highest.
  display(data[['name', 'artists','year']][2:10])

In [ ]:
#recommend_songs('Shape of You')
recommend_songs('Run')

In [ ]:
#recommend_songs('Love Someone')

In [ ]:
#recommend_songs('Love me like you do')